# 06 – Model Export
**AlchemiX – Launch26 Phase 2**

## Objectives
1. Confirm all three models are saved correctly
2. Generate inference examples for each model
3. Demonstrate the complete prediction pipeline
4. Generate deployment instructions

## Output Files
```
models/
├── congestion.joblib
├── trust.joblib
├── targeting.joblib
├── preprocessor_congestion.joblib
├── preprocessor_trust.joblib
├── preprocessor_targeting.joblib
└── model_metadata.json
```

In [1]:
import sys, pathlib, json
PROJECT_ROOT = pathlib.Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import warnings; warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import joblib

from ml.utils import (
    CONGESTION_MODEL_FILE, TRUST_MODEL_FILE, TARGETING_MODEL_FILE,
    MODELS_DIR,
)

print('Imports OK ✓')

Imports OK ✓


## 1. Verify Saved Models

In [2]:
model_files = [
    CONGESTION_MODEL_FILE,
    TRUST_MODEL_FILE,
    TARGETING_MODEL_FILE,
    MODELS_DIR / 'preprocessor_congestion.joblib',
    MODELS_DIR / 'preprocessor_trust.joblib',
    MODELS_DIR / 'preprocessor_targeting.joblib',
    MODELS_DIR / 'model_metadata.json',
]

print('Model files status:')
for f in model_files:
    exists = pathlib.Path(f).exists()
    size   = pathlib.Path(f).stat().st_size / 1024 if exists else 0
    status = '✓' if exists else '✗ MISSING'
    print(f'  {status}  {pathlib.Path(f).name:<45} {size:8.1f} KB')

Model files status:
  ✓  congestion.joblib                                724.0 KB
  ✓  trust.joblib                                     888.9 KB
  ✓  targeting.joblib                                   5.9 KB
  ✓  preprocessor_congestion.joblib                     3.4 KB
  ✓  preprocessor_trust.joblib                          3.3 KB
  ✓  preprocessor_targeting.joblib                      3.0 KB
  ✓  model_metadata.json                                1.1 KB


In [3]:
congestion_model  = joblib.load(CONGESTION_MODEL_FILE)
trust_model       = joblib.load(TRUST_MODEL_FILE)
targeting_model   = joblib.load(TARGETING_MODEL_FILE)
preprocessor_t    = joblib.load(MODELS_DIR / 'preprocessor_congestion.joblib')
preprocessor_te   = joblib.load(MODELS_DIR / 'preprocessor_trust.joblib')
preprocessor_i    = joblib.load(MODELS_DIR / 'preprocessor_targeting.joblib')
meta              = json.loads((MODELS_DIR / 'model_metadata.json').read_text())

print(f'Congestion model type : {type(congestion_model).__name__}')
print(f'Trust model type      : {type(trust_model).__name__}')
print(f'Targeting model type  : {type(targeting_model).__name__}')

Congestion model type : XGBRegressor
Trust model type      : LGBMRegressor
Targeting model type  : DecisionTreeClassifier


## 2. Inference Examples

These examples simulate what the Go AI Agent does at competition time.

In [4]:
def predict_congestion(
    link_id: str,
    load_units: float,
    load_ratio: float,
    status: str,
    hist_mean_load_ratio: float = 0.35,
    hist_std_load_ratio: float  = 0.20,
    hist_mean_load_units: float = 60.0,
) -> dict:
    """Predict congestion penalty (ms) from live network state."""
    planet_a, planet_b = link_id.split('-', 1)
    load_percentage    = load_ratio * 100
    near_capacity      = int(load_ratio >= 0.80)
    high_congestion    = int(load_ratio >= 0.90)
    status_ok          = int(status == 'ok')
    load_ratio_sq      = load_ratio ** 2
    load_units_x_ratio = load_units * load_ratio

    numeric_cols = meta['congestion']['numeric']
    cat_cols     = meta['congestion']['categorical']

    row_num = {
        'load_units': load_units, 'load_ratio': load_ratio,
        'load_percentage': load_percentage, 'load_ratio_sq': load_ratio_sq,
        'load_units_x_ratio': load_units_x_ratio,
        'near_capacity': near_capacity, 'high_congestion': high_congestion,
        'status_ok': status_ok,
        'hist_mean_load_ratio': hist_mean_load_ratio,
        'hist_std_load_ratio': hist_std_load_ratio,
        'hist_mean_load_units': hist_mean_load_units,
    }
    row_cat = {'planet_a': planet_a, 'planet_b': planet_b}
    X = pd.DataFrame([{**row_num, **row_cat}])[numeric_cols + cat_cols]
    X_proc = preprocessor_t.transform(X)
    penalty_ms = float(congestion_model.predict(X_proc)[0])
    return {'link_id': link_id, 'congestion_penalty_ms': round(penalty_ms, 2)}


def predict_trust(
    link_id: str,
    self_reported_latency_ms: float,
    measured_latency_ms: float,
    hist_mean_trust: float = 0.9,
    hist_std_trust: float  = 0.1,
    hist_mean_latency_diff: float = 5000.0,
) -> dict:
    """Predict trust score [0–1] from telemetry values."""
    planet_a, planet_b = link_id.split('-', 1)
    diff = measured_latency_ms - self_reported_latency_ms
    abs_diff = abs(diff)
    ratio = measured_latency_ms / max(self_reported_latency_ms, 1e-6)
    pct_error = abs_diff / max(measured_latency_ms, 1e-6) * 100

    numeric_cols = meta['trust']['numeric']
    cat_cols     = meta['trust']['categorical']

    row = {
        'self_reported_latency_ms': self_reported_latency_ms,
        'measured_latency_ms': measured_latency_ms,
        'latency_difference': diff,
        'absolute_difference': abs_diff,
        'latency_ratio': ratio,
        'percentage_error': pct_error,
        'hist_mean_trust': hist_mean_trust,
        'hist_std_trust': hist_std_trust,
        'hist_mean_latency_diff': hist_mean_latency_diff,
        'planet_a': planet_a, 'planet_b': planet_b,
    }
    X = pd.DataFrame([row])[numeric_cols + cat_cols]
    X_proc = preprocessor_te.transform(X)
    trust = float(np.clip(trust_model.predict(X_proc)[0], 0, 1))
    trust_penalty_ms = (1 - trust) * 50_000  # 50s max penalty for full distrust
    return {
        'link_id': link_id,
        'trust_score': round(trust, 4),
        'trust_penalty_ms': round(trust_penalty_ms, 2),
    }


def predict_targeting(
    link_id: str,
    traffic_share: float,
    hist_attack_rate: float = 0.10,
) -> dict:
    """Predict probability that Chimera jams this link."""
    planet_a, planet_b = link_id.split('-', 1)
    traffic_percentage = traffic_share * 100
    high_traffic       = int(traffic_share >= 0.20)
    relative_traffic   = traffic_share / 0.0833  # approx mean for 12 links

    numeric_cols = meta['targeting']['numeric']
    cat_cols     = meta['targeting']['categorical']

    row = {
        'traffic_share': traffic_share,
        'traffic_percentage': traffic_percentage,
        'high_traffic_share': high_traffic,
        'relative_traffic': relative_traffic,
        'hist_attack_rate': hist_attack_rate,
        'planet_a': planet_a, 'planet_b': planet_b,
    }
    X = pd.DataFrame([row])[numeric_cols + cat_cols]
    X_proc = preprocessor_i.transform(X)
    jam_prob    = float(targeting_model.predict_proba(X_proc)[0][1])
    jam_penalty = jam_prob * 200_000  # 200s max penalty for certain jam
    return {
        'link_id': link_id,
        'jam_probability': round(jam_prob, 4),
        'targeting_penalty_ms': round(jam_penalty, 2),
    }


def predict_true_cost(
    link_id: str,
    physics_latency_ms: float,
    **kwargs,
) -> dict:
    """Calculate the True Cost for one link sequentially."""
    congestion = predict_congestion(link_id, **{k: v for k, v in kwargs.items()
                                                 if k in ['load_units','load_ratio','status',
                                                          'hist_mean_load_ratio','hist_std_load_ratio','hist_mean_load_units']})
    trust      = predict_trust(link_id, **{k: v for k, v in kwargs.items()
                                            if k in ['self_reported_latency_ms','measured_latency_ms',
                                                     'hist_mean_trust','hist_std_trust','hist_mean_latency_diff']})
    targeting  = predict_targeting(link_id, **{k: v for k, v in kwargs.items()
                                               if k in ['traffic_share','hist_attack_rate']})

    true_cost = (
        physics_latency_ms
        + congestion['congestion_penalty_ms']
        + trust['trust_penalty_ms']
        + targeting['targeting_penalty_ms']
    )
    return {
        'link_id': link_id,
        'physics_latency_ms':    physics_latency_ms,
        'congestion_penalty_ms': congestion['congestion_penalty_ms'],
        'trust_score':           trust['trust_score'],
        'trust_penalty_ms':      trust['trust_penalty_ms'],
        'jam_probability':       targeting['jam_probability'],
        'targeting_penalty_ms':  targeting['targeting_penalty_ms'],
        'true_cost_ms':          round(true_cost, 2),
    }

print('Inference functions defined ✓')

Inference functions defined ✓


In [5]:
# ── Example 1: A healthy link ─────────────────────────────────────────────────
print('Example 1 – Healthy link (Aegis-Boreas)')
print('─' * 60)
result1 = predict_true_cost(
    link_id='Aegis-Boreas',
    physics_latency_ms=62_000,
    load_units=45.0, load_ratio=0.22, status='ok',
    self_reported_latency_ms=63_000, measured_latency_ms=64_000,
    traffic_share=0.05,
)
display(pd.DataFrame([result1]))

# ── Example 2: A congested, dishonest, high-traffic link ─────────────────────
print('\nExample 2 – Congested / dishonest link (Caelum-Elysium)')
print('─' * 60)
result2 = predict_true_cost(
    link_id='Caelum-Elysium',
    physics_latency_ms=180_000,
    load_units=160.0, load_ratio=0.88, status='ok',
    self_reported_latency_ms=50_000, measured_latency_ms=400_000,
    traffic_share=0.28,
)
display(pd.DataFrame([result2]))

print(f'\nDifference in True Cost: {result2["true_cost_ms"] - result1["true_cost_ms"]:,.0f} ms')
print('→ The AI Agent would prefer the Aegis-Boreas route.')

Example 1 – Healthy link (Aegis-Boreas)
────────────────────────────────────────────────────────────


,link_id,physics_latency_ms,congestion_penalty_ms,trust_score,trust_penalty_ms,jam_probability,targeting_penalty_ms,true_cost_ms
0,Aegis-Boreas,62000,154057.8,0.9844,779.46,0.0886,17721.52,234558.78



Example 2 – Congested / dishonest link (Caelum-Elysium)
────────────────────────────────────────────────────────────


,link_id,physics_latency_ms,congestion_penalty_ms,trust_score,trust_penalty_ms,jam_probability,targeting_penalty_ms,true_cost_ms
0,Caelum-Elysium,180000,1054072.12,0.5628,21861.63,0.1552,31046.93,1286980.68



Difference in True Cost: 1,052,422 ms
→ The AI Agent would prefer the Aegis-Boreas route.


## 3. Sequential Route Evaluation Example

In [6]:
# Simulate evaluating a route: Aegis → Boreas → Dawn → Caelum
# Each link is evaluated independently (Phase 2 requirement)
route_links = [
    ('Aegis-Boreas',  62_000, 45.0, 0.22, 'ok', 63_000, 64_000, 0.05),
    ('Boreas-Dawn',  95_000, 80.0, 0.40, 'ok', 96_000, 97_000, 0.09),
    ('Caelum-Dawn',  210_000, 85.0, 0.72, 'ok', 120_000, 420_000, 0.25),
]

records = []
total_true_cost = 0
for link_id, phys, lu, lr, st, sr, meas, ts in route_links:
    r = predict_true_cost(
        link_id=link_id, physics_latency_ms=phys,
        load_units=lu, load_ratio=lr, status=st,
        self_reported_latency_ms=sr, measured_latency_ms=meas,
        traffic_share=ts,
    )
    total_true_cost += r['true_cost_ms']
    records.append(r)

route_df = pd.DataFrame(records)
display(route_df)
print(f'\nTotal Route True Cost: {total_true_cost:,.0f} ms  ({total_true_cost/1000:.1f} s)')

,link_id,physics_latency_ms,congestion_penalty_ms,trust_score,trust_penalty_ms,jam_probability,targeting_penalty_ms,true_cost_ms
0,Aegis-Boreas,62000,154057.80,0.9844,779.46,0.0886,17721.52,234558.78
1,Boreas-Dawn,95000,222678.16,0.9895,525.52,0.0622,12443.10,330646.78
2,Caelum-Dawn,210000,692661.81,0.5921,20395.59,0.1552,31046.93,954104.33



Total Route True Cost: 1,519,310 ms  (1519.3 s)


## 4. Deployment Instructions

In [7]:
instructions = """
╔══════════════════════════════════════════════════════════════════╗
║              ALCHEMIX PHASE 2 — DEPLOYMENT CHECKLIST            ║
╠══════════════════════════════════════════════════════════════════╣
║                                                                  ║
║  1. Install requirements:                                        ║
║     pip install pandas scikit-learn xgboost lightgbm joblib      ║
║                                                                  ║
║  2. Required model files (models/ directory):                    ║
║     ✓ congestion.joblib                                          ║
║     ✓ trust.joblib                                               ║
║     ✓ targeting.joblib                                           ║
║     ✓ preprocessor_congestion.joblib                             ║
║     ✓ preprocessor_trust.joblib                                  ║
║     ✓ preprocessor_targeting.joblib                              ║
║     ✓ model_metadata.json                                        ║
║                                                                  ║
║  3. Environment variables:                                       ║
║     CHIMERA_API_URL=http://<host>/state   (live API endpoint)    ║
║     ORCHESTRATOR_URL=http://localhost:8080                       ║
║                                                                  ║
║  4. AI Agent (Go):                                               ║
║     The Go ai/ package calls models via HTTP inference service.  ║
║     Start inference server: python -m ml.inference_server        ║
║     Then run: wails dev                                          ║
║                                                                  ║
║  5. Competition day:                                             ║
║     - NEVER retrain on live API data                             ║
║     - Only feed live API values as inference inputs              ║
║     - Use GET /state for load, traffic_share, self_reported      ║
║                                                                  ║
╚══════════════════════════════════════════════════════════════════╝
"""
print(instructions)


╔══════════════════════════════════════════════════════════════════╗
║              ALCHEMIX PHASE 2 — DEPLOYMENT CHECKLIST            ║
╠══════════════════════════════════════════════════════════════════╣
║                                                                  ║
║  1. Install requirements:                                        ║
║     pip install pandas scikit-learn xgboost lightgbm joblib      ║
║                                                                  ║
║  2. Required model files (models/ directory):                    ║
║     ✓ congestion.joblib                                          ║
║     ✓ trust.joblib                                               ║
║     ✓ targeting.joblib                                           ║
║     ✓ preprocessor_congestion.joblib                             ║
║     ✓ preprocessor_trust.joblib                                  ║
║     ✓ preprocessor_targeting.joblib                              ║
║     ✓ model_metadata.json       

In [8]:
print('=' * 60)
print('  ALCHEMIX PHASE 2 — ALL NOTEBOOKS COMPLETE')
print('=' * 60)
print('  01 EDA                 ✓')
print('  02 Data Preprocessing  ✓')
print('  03 Feature Engineering ✓')
print('  04 Model Training      ✓')
print('  05 Model Evaluation    ✓')
print('  06 Model Export        ✓  ← you are here')
print('=' * 60)
print('\nNext steps:')
print('  → Implement ai/agent.go, ai/inference.go, ai/cost.go')
print('  → Connect to live GET /state API')
print('  → Run integration test with the Phase 1 router')

  ALCHEMIX PHASE 2 — ALL NOTEBOOKS COMPLETE
  01 EDA                 ✓
  02 Data Preprocessing  ✓
  03 Feature Engineering ✓
  04 Model Training      ✓
  05 Model Evaluation    ✓
  06 Model Export        ✓  ← you are here

Next steps:
  → Implement ai/agent.go, ai/inference.go, ai/cost.go
  → Connect to live GET /state API
  → Run integration test with the Phase 1 router
